In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import re
from PIL import Image


In [2]:
# Paths
image_folder = "path_to_images"  # Update this with your images folder
csv_path = "indiana_projections.csv"  # Update this with your CSV file path

# Load the dataset
df = pd.read_csv(csv_path)

In [3]:
# Function to check if an image is corrupted
def is_corrupted(image_path):
    try:
        img = Image.open(image_path)
        img.verify()  # Check for corruption
        return False  # Image is fine
    except Exception:
        return True  # Corrupted image detected


In [4]:
# Function to check if an image is blank or too noisy
def is_noisy_or_blank(image_path, threshold=0.02):
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return True  # Image is unreadable or missing
        
        # Calculate standard deviation of pixel intensity
        std_dev = np.std(img)
        return std_dev < threshold  # If very low contrast, consider blank/noisy
    except Exception:
        return True  # If any error occurs, consider it bad
       

In [5]:
# Iterate through images and remove corrupted or blank images
valid_images = []
for img_name in df['images/images_normalized']:  # Update with the correct column name
    img_path = os.path.join(image_folder, img_name)
    if os.path.exists(img_path) and not is_corrupted(img_path) and not is_noisy_or_blank(img_path):
        valid_images.append(img_name)

# Update the dataset to keep only valid images
df = df[df['image_path_column'].isin(valid_images)]  # Filter dataset
df.to_csv("cleaned_dataset.csv", index=False)

print(f"✅ Dataset cleaned! Kept {len(df)} valid images.")


KeyError: 'images/images_normalized'

In [6]:
import os
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths (Update these with actual paths)
csv_file = "C:/Users/ag331/OneDrive/Desktop/med_project/archive (2)/merged_indiana_dataset.csv"  # CSV containing report text and image mappings
image_dir = "images/images_normalized"  # Folder containing CXR images
output_csv = "processed_dataset.csv"  # Output file


In [7]:
import pandas as pd

# Load the CSV files (Update with the actual file paths if needed)
projections_path = "standardized_reports_final (1).csv"  # CSV with image filenames & projections
reports_path = "indiana_reports.csv"  # CSV with radiology reports

# Read CSV files into Pandas DataFrames
df_projections = pd.read_csv(projections_path)
df_reports = pd.read_csv(reports_path)

# Merge datasets using `uid` to align images with their corresponding reports
df_merged = df_projections.merge(df_reports, on="uid", how="inner")

# Save the merged dataset
merged_csv_path = "merged_indiana_dataset.csv"
df_merged.to_csv(merged_csv_path, index=False)

print(f"✅ Merging complete! Merged dataset saved as: {merged_csv_path}")


✅ Merging complete! Merged dataset saved as: merged_indiana_dataset.csv


In [8]:

# Load the dataset
df = pd.read_csv(merged_csv_path)

# Ensure required columns exist
required_columns = {"uid", "filename", "projection", "findings", "impression","indication","Problems","comparison","MeSH"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"CSV file is missing required columns: {required_columns - set(df.columns)}")


In [12]:

# Store processed data
processed_data = []

# Group images by report ID (`uid`)
grouped = df.groupby("uid")


In [14]:

# Process each report
for uid, group in grouped:
    # Extract findings and impressions
    findings = group["findings"].iloc[0]
    impression = group["impression"].iloc[0]
    Problems = group["Problems"].iloc[0]
    comparison = group["comparison"].iloc[0]
    MeSH = group["MeSH"].iloc[0]
    indication = group["indication"].iloc[0]

    # Get all associated images
    images = list(zip(group["filename"], group["projection"],))

    # Separate into Frontal and Lateral images
    frontal_images = [img for img, proj in images if proj == "Frontal"]
    lateral_images = [img for img, proj in images if proj == "Lateral"]

    # Select image pairs (or duplicate single images)
    if len(frontal_images) == 1 and len(lateral_images) == 1:
        selected_images = [(frontal_images[0], "Frontal"), (lateral_images[0], "Lateral")]

    elif len(frontal_images) == 0 or len(lateral_images) == 0:
        # Duplicate single image to create a pair
        single_image = frontal_images[0] if frontal_images else lateral_images[0]
        selected_images = [(single_image, "Frontal"), (single_image, "Lateral")]

    elif len(frontal_images) + len(lateral_images) == 3:
        # Pick one Frontal and one Lateral, discard the extra
        selected_images = [(frontal_images[0], "Frontal"), (lateral_images[0], "Lateral")]

    elif len(frontal_images) >= 2 and len(lateral_images) >= 2:
        # Pick one Frontal and one Lateral from four-image reports
        selected_images = [(frontal_images[0], "Frontal"), (lateral_images[0], "Lateral")]

    else:
        continue  # Skip cases with no images

    # Store processed data
    for img, proj in selected_images:
        img_path = os.path.join(image_dir, img)
        height, width = get_image_metadata(img_path)
        processed_data.append({
            "uid": uid,
            "filename": img,
            "projection": proj,
            "findings": findings,
            "impression": impression,
            "Problems":Problems,
            "MeSH":MeSH,
            "comparison":comparison,
            "indication":indication
        })

# Convert processed data to a DataFrame
processed_df = pd.DataFrame(processed_data)

# Save to CSV
processed_df.to_csv(output_csv, index=False)

print(f"✅ Preprocessing complete! Processed dataset saved to {output_csv}")


✅ Preprocessing complete! Processed dataset saved to processed_dataset.csv
